# TYPE-EXP-001 — FTR Vehicle Type Classifier

Amaç: FTR `arac_bilgisi.tip` alanı için dedicated 7 sınıflı araç tipi classifier eğitmek.

FTR tip sınıfları:

```text
sedan, suv, hatchback, pickup, minibus, panelvan, kamyon
```

Bu notebook neden gerekli?

- `VATTR-EXP-001` hız için body/dimension prior olarak kullanıldı; final FTR tip modeli değildir.
- `VEHINFO-EXP-001` OpenVINO + VATTR fusion sonucunda tip tahmini düşük güvenli kaldı.
- FTR output'u allowed label set ile birebir uyumlu tek `tip` değeri bekliyor.

Birincil otomatik kaynak:

- Stanford Cars Kaggle mirror: https://www.kaggle.com/datasets/eduardo4jesus/stanford-cars-dataset

Opsiyonel/ek kaynaklar:

- Manual FTR folder dataset: kendi klasörlerinizi `sedan/suv/hatchback/pickup/minibus/panelvan/kamyon` olarak koyabilirsiniz.
- BoxCars-derived FTR folder dataset: BoxCars'tan dönüştürülmüş crop'lar varsa aynı manual klasör mantığıyla eklenebilir.

Önemli: Stanford Cars model isimlerinden tip parsing yapılır. Mapping şüpheli ise örnekler eğitimden atılır; yanlış label ile eğitim yapmamak için agresif mapping yerine konservatif mapping tercih edilir.


## Senin Yapman Gerekenler

1. Colab Runtime type: GPU seç. L4 veya A100 iyi; T4 de çalışır.
2. Colab Secrets içine Kaggle bilgilerini ekle:
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`
   - alternatif küçük harfli isimler de desteklenir: `kaggle_username`, `kaggle_key`
3. Stanford Cars Kaggle erişimi çalışmazsa manuel fallback:
   - Stanford Cars zip dosyasını indir.
   - Drive içine şuraya koy:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/stanford_cars/
```

4. Eğer bazı FTR tip sınıfları düşük veya eksik çıkarsa manual ek veri klasörü kullan:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/sedan/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/suv/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/hatchback/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/pickup/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/minibus/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/panelvan/*.jpg
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/manual/kamyon/*.jpg
```

5. Varsayılan ağır run için `SMOKE_MODE = False` bırak. Hızlı test için `SMOKE_MODE = True` yapabilirsin.

Başarılı run çıktıları Drive altında oluşur:

```text
/content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_type/
/content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_type/TYPE-EXP-001/
/content/drive/MyDrive/anomali-road-safety-ai/datasets/type_exp_001/metadata/
```


In [ ]:
# Cell 1 — Runtime setup + dependencies
import os
import sys
import json
import math
import time
import random
import shutil
import zipfile
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

def run_cmd(cmd, check=True, capture_output=False):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=check, text=True, capture_output=capture_output)

if IN_COLAB:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle', 'scikit-learn', 'pandas', 'matplotlib', 'seaborn', 'tqdm', 'scipy'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

print('IN_COLAB:', IN_COLAB)
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 — Drive mount + configuration
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai') if IN_COLAB else Path.cwd()
LOCAL_ROOT = Path('/content/anomali-road-safety-ai-work') if IN_COLAB else Path.cwd() / '.local_colab_work'
EXP_ID = 'TYPE-EXP-001'

DRIVE_DATA_ROOT = PROJECT_ROOT / 'datasets' / 'type_exp_001'
DRIVE_STANFORD_ROOT = DRIVE_DATA_ROOT / 'stanford_cars'
DRIVE_MANUAL_ROOT = DRIVE_DATA_ROOT / 'manual'
DRIVE_METADATA_ROOT = DRIVE_DATA_ROOT / 'metadata'
DRIVE_RUN_ROOT = PROJECT_ROOT / 'runs' / 'vehicle_type' / EXP_ID
DRIVE_CKPT_ROOT = PROJECT_ROOT / 'models' / 'checkpoints' / 'vehicle_type'

LOCAL_DATA_ROOT = LOCAL_ROOT / 'datasets' / 'type_exp_001'
LOCAL_STANFORD_ROOT = LOCAL_DATA_ROOT / 'stanford_cars'
LOCAL_STANFORD_EXTRACT = LOCAL_STANFORD_ROOT / 'extracted'
LOCAL_MANUAL_ROOT = LOCAL_DATA_ROOT / 'manual'

for p in [DRIVE_STANFORD_ROOT, DRIVE_MANUAL_ROOT, DRIVE_METADATA_ROOT, DRIVE_RUN_ROOT, DRIVE_CKPT_ROOT, LOCAL_STANFORD_EXTRACT, LOCAL_MANUAL_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

KAGGLE_STANFORD_SLUGS = [
    'eduardo4jesus/stanford-cars-dataset',
    'jessicali9530/stanford-cars-dataset',
]
DOWNLOAD_METHOD = 'kaggle'  # kaggle | manual | skip
ENABLE_STANFORD = True
ENABLE_MANUAL_FOLDERS = True
USE_EXISTING_EXTRACT = True
SMOKE_MODE = False
MAX_IMAGES_PER_CLASS = 160 if SMOKE_MODE else None
MIN_IMAGES_PER_CLASS_WARN = 40
REQUIRE_ALL_FTR_CLASSES = False  # Set True if you want the notebook to stop when a class is missing.
GROUP_SPLIT_BY_SOURCE_CLASS = True

IMAGE_SIZE = 224
BATCH_SIZE = 48 if torch.cuda.is_available() else 16
NUM_WORKERS = 0  # Colab Python 3.12 can emit noisy DataLoader worker shutdown warnings.
EPOCHS = 4 if SMOKE_MODE else 18
PATIENCE = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
RUN_MOBILENETV3_LARGE = True
RUN_EFFICIENTNET_B0 = True
FREEZE_EPOCHS = 2 if not SMOKE_MODE else 1
UNFREEZE_BACKBONE = True

FTR_TYPE_LABELS = ['sedan', 'suv', 'hatchback', 'pickup', 'minibus', 'panelvan', 'kamyon']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('DRIVE_STANFORD_ROOT:', DRIVE_STANFORD_ROOT)
print('DRIVE_MANUAL_ROOT:', DRIVE_MANUAL_ROOT)
print('EPOCHS:', EPOCHS, 'BATCH_SIZE:', BATCH_SIZE, 'SMOKE_MODE:', SMOKE_MODE)


In [ ]:
# Cell 3 — Kaggle auth + dataset acquisition
import getpass

def setup_kaggle_credentials():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'

    username = os.environ.get('KAGGLE_USERNAME') or os.environ.get('kaggle_username')
    key = os.environ.get('KAGGLE_KEY') or os.environ.get('kaggle_key')

    if IN_COLAB and (not username or not key):
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME') or userdata.get('kaggle_username')
            key = key or userdata.get('KAGGLE_KEY') or userdata.get('kaggle_key')
        except Exception as exc:
            print('Colab Secrets read skipped/failed:', exc)

    if not username or not key:
        if DOWNLOAD_METHOD == 'kaggle':
            print('Kaggle credentials are missing. You can enter them now, or set DOWNLOAD_METHOD="manual".')
            username = input('Kaggle username: ').strip()
            key = getpass.getpass('Kaggle key: ').strip()
        else:
            return None

    kaggle_json.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
    os.chmod(kaggle_json, 0o600)
    print('Kaggle credentials ready:', kaggle_json)
    return kaggle_json


def find_zip_candidates(root):
    candidates = sorted(root.glob('*.zip'))
    out = []
    seen = set()
    for p in candidates:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def dataset_has_images(root):
    return any(p.suffix.lower() in IMAGE_EXTS for p in root.rglob('*') if p.is_file())


def extract_zip(zip_path, target_root):
    target_root.mkdir(parents=True, exist_ok=True)
    marker = target_root / (zip_path.name + '.extracted.marker')
    if marker.exists() and dataset_has_images(target_root):
        print('Archive already extracted:', zip_path)
        return
    print('Extracting:', zip_path, '->', target_root)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_root)
    marker.write_text(str(time.time()), encoding='utf-8')


def download_stanford_if_needed():
    if not ENABLE_STANFORD:
        print('Stanford source disabled.')
        return
    if USE_EXISTING_EXTRACT and dataset_has_images(LOCAL_STANFORD_EXTRACT):
        print('Local extracted Stanford images already exist:', LOCAL_STANFORD_EXTRACT)
        return

    zip_candidates = find_zip_candidates(DRIVE_STANFORD_ROOT)
    if not zip_candidates and DOWNLOAD_METHOD == 'kaggle':
        setup_kaggle_credentials()
        last_error = None
        for slug in KAGGLE_STANFORD_SLUGS:
            try:
                print('Downloading Stanford Cars from Kaggle:', slug)
                run_cmd(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(DRIVE_STANFORD_ROOT)])
                zip_candidates = find_zip_candidates(DRIVE_STANFORD_ROOT)
                if zip_candidates:
                    break
            except Exception as exc:
                last_error = exc
                print('Kaggle download failed for', slug, ':', exc)
        if not zip_candidates and last_error:
            print('Last Kaggle error:', last_error)

    if not zip_candidates:
        raise FileNotFoundError(
            'No Stanford Cars zip found. Put the dataset zip under:\n'
            f'  {DRIVE_STANFORD_ROOT}\n'
            'or set valid Kaggle credentials and DOWNLOAD_METHOD="kaggle".'
        )

    print('Zip candidates:')
    for z in zip_candidates:
        print(' ', z, 'size MB:', round(z.stat().st_size / (1024*1024), 2))
    for z in zip_candidates:
        extract_zip(z, LOCAL_STANFORD_EXTRACT)

    if not dataset_has_images(LOCAL_STANFORD_EXTRACT):
        raise RuntimeError('Extraction finished but no Stanford images were found under ' + str(LOCAL_STANFORD_EXTRACT))


def sync_manual_folders():
    if not ENABLE_MANUAL_FOLDERS:
        return
    if not DRIVE_MANUAL_ROOT.exists() or not any(DRIVE_MANUAL_ROOT.rglob('*')):
        print('Manual folder is empty/skipped:', DRIVE_MANUAL_ROOT)
        return
    print('Copying manual folders to local workdir:', DRIVE_MANUAL_ROOT, '->', LOCAL_MANUAL_ROOT)
    if LOCAL_MANUAL_ROOT.exists():
        shutil.rmtree(LOCAL_MANUAL_ROOT)
    shutil.copytree(DRIVE_MANUAL_ROOT, LOCAL_MANUAL_ROOT)

if ENABLE_STANFORD:
    download_stanford_if_needed()
sync_manual_folders()

print('Stanford local image count:', sum(1 for p in LOCAL_STANFORD_EXTRACT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS))
print('Manual local image count:', sum(1 for p in LOCAL_MANUAL_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS))


In [ ]:
# Cell 4 — FTR type mapping helpers
import re

def normalize_text(value):
    value = str(value).lower().strip()
    replace = {
        'ı': 'i', 'İ': 'i', 'ş': 's', 'Ş': 's', 'ğ': 'g', 'Ğ': 'g',
        'ü': 'u', 'Ü': 'u', 'ö': 'o', 'Ö': 'o', 'ç': 'c', 'Ç': 'c',
    }
    for src, dst in replace.items():
        value = value.replace(src, dst)
    value = re.sub(r'[^a-z0-9]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return value

# Explicit FTR or high-confidence alias tokens.
DIRECT_ALIASES = {
    'sedan': 'sedan', 'saloon': 'sedan',
    'suv': 'suv', 'crossover': 'suv',
    'hatchback': 'hatchback', 'hatch': 'hatchback',
    'pickup': 'pickup', 'pick_up': 'pickup', 'pickup_truck': 'pickup',
    'minibus': 'minibus', 'minivan': 'minibus', 'mini_van': 'minibus', 'shuttle': 'minibus',
    'panelvan': 'panelvan', 'panel_van': 'panelvan', 'cargo_van': 'panelvan',
    'kamyon': 'kamyon', 'truck': 'kamyon', 'lorry': 'kamyon', 'semi': 'kamyon', 'tractor_trailer': 'kamyon',
}

# Model-name heuristics for Stanford-style labels. Order matters: pickup before kamyon because "cab" labels are pickup-like.
TYPE_RULES = [
    ('pickup', ['pickup', 'pick_up', 'regular_cab', 'crew_cab', 'extended_cab', 'supercab', 'super_crew', 'club_cab', 'quad_cab', 'f_150', 'f150', 'silverado', 'dakota', 'ram_1500', 'ranger', 'tacoma', 'tundra', 'canyon', 'avalanche']),
    ('kamyon', ['lorry', 'semi', 'tractor_trailer', 'box_truck', 'dump_truck', 'truck']),
    ('minibus', ['minivan', 'mini_van', 'odyssey', 'caravan', 'freestar', 'passenger_van', 'shuttle', 'minibus']),
    ('panelvan', ['cargo_van', 'panel_van', 'sprinter', 'express_cargo', 'savana', 'e_series', 'van']),
    ('suv', ['suv', 'crossover', 'hummer', 'wrangler', 'cherokee', 'patriot', 'compass', 'liberty', 'durango', 'trailblazer', 'expedition', 'terrain', 'yukon', 'tahoe', 'suburban', 'range_rover', 'land_rover', 'rav4', 'forester', 'veracruz', 'tucson', 'santa_fe', 'acadia', 'enclave', 'x5', 'x6', 'qx56', 'ascender']),
    ('hatchback', ['hatchback', 'hatch', 'compact', 'zdx', 'caliber', 'hhr', 'fiesta_hatch']),
    ('sedan', ['sedan', 'saloon', 'accord_sedan', 'elantra', 'sonata', 'camry', 'corolla', 'civic_sedan', 'impala', 'malibu', 'azera', 'genesis_sedan', 'rl', 'tl']),
]

UNSUPPORTED_TOKENS = ['coupe', 'convertible', 'roadster', 'wagon']

# If a specific source class is known to map correctly, add it here.
# Key should be normalized source class name.
SOURCE_CLASS_OVERRIDES = {
    # 'ford_e_series_wagon_van_2012': 'panelvan',
}


def infer_ftr_type_from_source_class(source_class):
    norm = normalize_text(source_class)
    if norm in SOURCE_CLASS_OVERRIDES:
        return SOURCE_CLASS_OVERRIDES[norm], 'override'
    tokens = set(norm.split('_'))
    for token, label in DIRECT_ALIASES.items():
        if token in tokens or token == norm:
            return label, 'direct_alias:' + token
    for label, needles in TYPE_RULES:
        for needle in needles:
            if needle in norm:
                return label, 'rule:' + needle
    # Avoid silently mapping clear non-FTR body styles into a wrong class.
    for needle in UNSUPPORTED_TOKENS:
        if needle in norm:
            return None, 'unsupported_body_style:' + needle
    return None, 'no_reliable_type_token'


def collect_class_folder_records(root, source_dataset):
    records = []
    skipped = []
    if not root.exists():
        return records, skipped
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:
            continue
        rel_parts = path.relative_to(root).parts
        if len(rel_parts) < 2:
            source_class = path.parent.name
        else:
            # In common datasets, the immediate parent is the class folder.
            source_class = path.parent.name
        label, reason = infer_ftr_type_from_source_class(source_class)
        row = {
            'image_path': str(path),
            'source_dataset': source_dataset,
            'source_class': source_class,
            'mapping_reason': reason,
            'group_id': f'{source_dataset}:{normalize_text(source_class)}',
        }
        if label in FTR_TYPE_LABELS:
            row['ftr_type'] = label
            records.append(row)
        else:
            skipped.append(row)
    return records, skipped

# Quick sanity test for important mappings.
for sample in ['Honda Accord Sedan 2012', 'Dodge Durango SUV 2012', 'Ford F-150 Regular Cab 2012', 'Dodge Caravan Minivan 1997', 'Mercedes-Benz Sprinter Van 2012', 'Freightliner Box Truck', 'Acura Integra Type R 2001']:
    print(sample, '->', infer_ftr_type_from_source_class(sample))


In [ ]:
# Cell 5 — Build metadata from Stanford + manual FTR folders
all_records = []
all_skipped = []

if ENABLE_STANFORD and LOCAL_STANFORD_EXTRACT.exists():
    rec, skip = collect_class_folder_records(LOCAL_STANFORD_EXTRACT, 'stanford_cars')
    all_records.extend(rec)
    all_skipped.extend(skip)
    print('Stanford records:', len(rec), 'skipped:', len(skip))

if ENABLE_MANUAL_FOLDERS and LOCAL_MANUAL_ROOT.exists():
    rec, skip = collect_class_folder_records(LOCAL_MANUAL_ROOT, 'manual_ftr_type')
    all_records.extend(rec)
    all_skipped.extend(skip)
    print('Manual records:', len(rec), 'skipped:', len(skip))

metadata_df = pd.DataFrame(all_records).drop_duplicates(subset=['image_path']).reset_index(drop=True)
skipped_df = pd.DataFrame(all_skipped).drop_duplicates(subset=['image_path']).reset_index(drop=True) if all_skipped else pd.DataFrame()

if metadata_df.empty:
    if not skipped_df.empty:
        print('Skipped sample:')
        display(skipped_df.head(50))
    raise RuntimeError('No FTR type-labelled images were inferred. Add manual folders or adjust mapping rules.')

if MAX_IMAGES_PER_CLASS is not None:
    metadata_df = (
        metadata_df.groupby('ftr_type', group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_IMAGES_PER_CLASS), random_state=SEED))
        .reset_index(drop=True)
    )

class_counts = metadata_df['ftr_type'].value_counts().reindex(FTR_TYPE_LABELS, fill_value=0)
print('Mapped images:', len(metadata_df))
print('Skipped images:', len(skipped_df))
print(class_counts)
print('\nBy source dataset:')
print(metadata_df.groupby(['source_dataset', 'ftr_type']).size().unstack(fill_value=0).reindex(columns=FTR_TYPE_LABELS, fill_value=0))

low_or_missing = class_counts[class_counts < MIN_IMAGES_PER_CLASS_WARN]
if len(low_or_missing):
    print('\nWARNING: low/missing FTR type classes. Add manual/BoxCars/CompCars data for these labels before final FTR promotion:')
    print(low_or_missing)
    if REQUIRE_ALL_FTR_CLASSES:
        raise RuntimeError('At least one FTR class is below MIN_IMAGES_PER_CLASS_WARN and REQUIRE_ALL_FTR_CLASSES=True')

DRIVE_METADATA_ROOT.mkdir(parents=True, exist_ok=True)
metadata_csv = DRIVE_METADATA_ROOT / 'type_exp_001_ftr_type_metadata.csv'
skipped_csv = DRIVE_METADATA_ROOT / 'type_exp_001_skipped_images.csv'
metadata_df.to_csv(metadata_csv, index=False)
skipped_df.to_csv(skipped_csv, index=False)
print('metadata_csv:', metadata_csv)
print('skipped_csv:', skipped_csv)


In [ ]:
# Cell 6 — Train/val/test split
usable_df = metadata_df.copy()
valid_classes = usable_df['ftr_type'].value_counts()
valid_classes = valid_classes[valid_classes >= 3].index.tolist()
usable_df = usable_df[usable_df['ftr_type'].isin(valid_classes)].reset_index(drop=True)

if len(valid_classes) < 2:
    raise RuntimeError('Need at least 2 type classes with >=3 images for split/training.')

# Group split reduces same source-class leakage. If it drops classes from val/test, fall back to stratified image split.
def try_group_split(df):
    groups = df['group_id'].astype(str).values
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    train_val_idx, test_idx = next(gss1.split(df, df['ftr_type'], groups=groups))
    train_val = df.iloc[train_val_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)
    tv_groups = train_val['group_id'].astype(str).values
    train_idx, val_idx = next(gss2.split(train_val, train_val['ftr_type'], groups=tv_groups))
    train = train_val.iloc[train_idx].reset_index(drop=True)
    val = train_val.iloc[val_idx].reset_index(drop=True)
    return train, val, test


def stratified_image_split(df):
    train_val, test = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df['ftr_type'])
    train, val = train_test_split(train_val, test_size=0.1765, random_state=SEED, stratify=train_val['ftr_type'])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

if GROUP_SPLIT_BY_SOURCE_CLASS:
    try:
        train_df, val_df, test_df = try_group_split(usable_df)
        required = set(usable_df['ftr_type'].unique())
        if not required.issubset(set(train_df['ftr_type'])):
            raise RuntimeError('group split dropped classes from train')
        print('Using group split by source_class.')
    except Exception as exc:
        print('Group split fallback to stratified image split:', exc)
        train_df, val_df, test_df = stratified_image_split(usable_df)
else:
    train_df, val_df, test_df = stratified_image_split(usable_df)

for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df = df.copy().reset_index(drop=True)
    df['split'] = split_name
    out = DRIVE_METADATA_ROOT / f'type_exp_001_{split_name}.csv'
    df.to_csv(out, index=False)
    print('\n', split_name, len(df), out)
    print(df['ftr_type'].value_counts().reindex(FTR_TYPE_LABELS, fill_value=0))

split_df = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
], ignore_index=True)
split_csv = DRIVE_METADATA_ROOT / 'type_exp_001_split.csv'
split_df.to_csv(split_csv, index=False)
print('split_csv:', split_csv)


In [ ]:
# Cell 7 — Dataset and dataloaders
label_to_id = {label: idx for idx, label in enumerate(FTR_TYPE_LABELS)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
NUM_CLASSES = len(FTR_TYPE_LABELS)

class VehicleTypeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row['image_path'])
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = label_to_id[row['ftr_type']]
        return image, label, str(path)

train_tf = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.72, 1.0), ratio=(0.82, 1.22)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.16, contrast=0.16, saturation=0.10, hue=0.015),
    T.RandomRotation(degrees=4),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(VehicleTypeDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(VehicleTypeDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(VehicleTypeDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

class_counts = train_df['ftr_type'].value_counts().reindex(FTR_TYPE_LABELS, fill_value=0)
weight_counts = class_counts.replace(0, 1)
weights = (weight_counts.sum() / (len(FTR_TYPE_LABELS) * weight_counts)).astype(float)
# Missing labels cannot be learned; keep finite weight but report them.
class_weights = torch.tensor(weights.values, dtype=torch.float32)
print('class_counts_train:', class_counts.to_dict())
print('class_weights:', dict(zip(FTR_TYPE_LABELS, class_weights.tolist())))
print('train batches:', len(train_loader), 'val batches:', len(val_loader), 'test batches:', len(test_loader))


In [ ]:
# Cell 8 — Model helpers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

def build_model(backbone):
    if backbone == 'mobilenet_v3_large':
        weights = models.MobileNet_V3_Large_Weights.DEFAULT
        model = models.mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
    elif backbone == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
    else:
        raise ValueError('Unsupported backbone: ' + backbone)
    return model


def set_backbone_trainable(model, trainable):
    for name, param in model.named_parameters():
        if 'classifier' in name:
            param.requires_grad = True
        else:
            param.requires_grad = trainable


def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels, all_probs, all_paths = [], [], [], []
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    loss_total, n = 0.0, 0
    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            loss_total += float(loss.item()) * images.size(0)
            n += images.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_paths.extend(paths)
    return {
        'loss': loss_total / max(n, 1),
        'accuracy': accuracy_score(all_labels, all_preds) if all_labels else 0.0,
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0) if all_labels else 0.0,
        'labels': all_labels,
        'preds': all_preds,
        'probs': all_probs,
        'paths': all_paths,
    }


def train_one_backbone(backbone):
    print('\n=== Training', backbone, '===')
    model = build_model(backbone).to(device)
    set_backbone_trainable(model, trainable=False)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))

    ckpt_path = DRIVE_CKPT_ROOT / f'{EXP_ID}-{backbone}-best.pth'
    history = []
    best_val_f1 = -1.0
    best_epoch = -1
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_EPOCHS + 1 and UNFREEZE_BACKBONE:
            print('Unfreezing backbone at epoch', epoch)
            set_backbone_trainable(model, trainable=True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE * 0.35, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS - epoch + 1, 1))

        model.train()
        running_loss = 0.0
        seen = 0
        for images, labels, _ in tqdm(train_loader, desc=f'{backbone} epoch {epoch}/{EPOCHS}'):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * images.size(0)
            seen += images.size(0)
        scheduler.step()

        val_metrics = evaluate_model(model, val_loader)
        row = {
            'backbone': backbone,
            'epoch': epoch,
            'train_loss': running_loss / max(seen, 1),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        print(row)

        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            bad_epochs = 0
            torch.save({
                'experiment_id': EXP_ID,
                'backbone': backbone,
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'label_to_id': label_to_id,
                'id_to_label': id_to_label,
                'image_size': IMAGE_SIZE,
                'val_macro_f1': best_val_f1,
                'val_accuracy': val_metrics['accuracy'],
                'ftr_type_labels': FTR_TYPE_LABELS,
                'source_dataset': 'Stanford Cars + optional manual/BoxCars-derived folders',
                'mapping_rules_version': 'type_exp_001_v1',
            }, ckpt_path)
            print('Saved best checkpoint:', ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print('Early stopping at epoch', epoch)
                break

    hist_df = pd.DataFrame(history)
    hist_path = DRIVE_RUN_ROOT / f'{EXP_ID}-{backbone}-history.csv'
    DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
    hist_df.to_csv(hist_path, index=False)
    return {'backbone': backbone, 'best_val_macro_f1': best_val_f1, 'best_epoch': best_epoch, 'checkpoint': str(ckpt_path), 'history_csv': str(hist_path)}


In [ ]:
# Cell 9 — Train selected backbones
summaries = []
if RUN_MOBILENETV3_LARGE:
    summaries.append(train_one_backbone('mobilenet_v3_large'))
if RUN_EFFICIENTNET_B0:
    summaries.append(train_one_backbone('efficientnet_b0'))

summary_df = pd.DataFrame(summaries).sort_values('best_val_macro_f1', ascending=False).reset_index(drop=True)
summary_path = DRIVE_RUN_ROOT / f'{EXP_ID}-backbone_summary.csv'
summary_df.to_csv(summary_path, index=False)
print('Backbone summary:')
display(summary_df)
print('summary_path:', summary_path)
BEST_BACKBONE = summary_df.iloc[0]['backbone']
BEST_CHECKPOINT = Path(summary_df.iloc[0]['checkpoint'])
print('BEST_BACKBONE:', BEST_BACKBONE)
print('BEST_CHECKPOINT:', BEST_CHECKPOINT)


In [ ]:
# Cell 10 — Test evaluation, confusion matrix, and exports
def load_checkpoint_model(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model = build_model(ckpt['backbone']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt

best_model, best_ckpt = load_checkpoint_model(BEST_CHECKPOINT)
val_eval = evaluate_model(best_model, val_loader)
test_eval = evaluate_model(best_model, test_loader)

print('Validation macro-F1:', val_eval['macro_f1'], 'acc:', val_eval['accuracy'])
print('Test macro-F1:', test_eval['macro_f1'], 'acc:', test_eval['accuracy'])

report_dict = classification_report(
    test_eval['labels'],
    test_eval['preds'],
    labels=list(range(NUM_CLASSES)),
    target_names=FTR_TYPE_LABELS,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
report_csv = DRIVE_RUN_ROOT / f'{EXP_ID}-test_classification_report.csv'
report_df.to_csv(report_csv)
display(report_df)

cm = confusion_matrix(test_eval['labels'], test_eval['preds'], labels=list(range(NUM_CLASSES)))
cm_csv = DRIVE_RUN_ROOT / f'{EXP_ID}-test_confusion_matrix.csv'
pd.DataFrame(cm, index=FTR_TYPE_LABELS, columns=FTR_TYPE_LABELS).to_csv(cm_csv)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=FTR_TYPE_LABELS, yticklabels=FTR_TYPE_LABELS, cmap='Greys')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{EXP_ID} Test Confusion Matrix — {BEST_BACKBONE}')
plt.tight_layout()
cm_png = DRIVE_RUN_ROOT / f'{EXP_ID}-test_confusion_matrix.png'
plt.savefig(cm_png, dpi=180)
plt.show()

pred_rows = []
for label_id, pred_id, probs, path in zip(test_eval['labels'], test_eval['preds'], test_eval['probs'], test_eval['paths']):
    top_idx = int(np.argmax(probs))
    second_idx = int(np.argsort(probs)[-2]) if len(probs) > 1 else top_idx
    pred_rows.append({
        'image_path': path,
        'true_label': id_to_label[int(label_id)],
        'pred_label': id_to_label[int(pred_id)],
        'confidence': float(probs[top_idx]),
        'margin': float(probs[top_idx] - probs[second_idx]),
        'correct': int(label_id == pred_id),
    })
pred_df = pd.DataFrame(pred_rows)
pred_csv = DRIVE_RUN_ROOT / f'{EXP_ID}-test_predictions.csv'
pred_df.to_csv(pred_csv, index=False)

label_map_path = DRIVE_CKPT_ROOT / f'{EXP_ID}-label-map.json'
label_map = {
    'experiment_id': EXP_ID,
    'label_to_id': label_to_id,
    'id_to_label': {str(k): v for k, v in id_to_label.items()},
    'ftr_type_labels': FTR_TYPE_LABELS,
    'image_size': IMAGE_SIZE,
    'best_backbone': BEST_BACKBONE,
    'checkpoint': str(BEST_CHECKPOINT),
    'mapping_rules_version': 'type_exp_001_v1',
    'source_class_overrides': SOURCE_CLASS_OVERRIDES,
}
label_map_path.write_text(json.dumps(label_map, indent=2, ensure_ascii=False), encoding='utf-8')

summary_json = DRIVE_RUN_ROOT / f'{EXP_ID}-summary.json'
summary = {
    'experiment_id': EXP_ID,
    'best_backbone': BEST_BACKBONE,
    'best_checkpoint': str(BEST_CHECKPOINT),
    'label_map': str(label_map_path),
    'metadata_csv': str(metadata_csv),
    'split_csv': str(split_csv),
    'class_counts': metadata_df['ftr_type'].value_counts().reindex(FTR_TYPE_LABELS, fill_value=0).to_dict(),
    'train_count': int(len(train_df)),
    'val_count': int(len(val_df)),
    'test_count': int(len(test_df)),
    'val_accuracy': float(val_eval['accuracy']),
    'val_macro_f1': float(val_eval['macro_f1']),
    'test_accuracy': float(test_eval['accuracy']),
    'test_macro_f1': float(test_eval['macro_f1']),
    'low_or_missing_classes': class_counts[class_counts < MIN_IMAGES_PER_CLASS_WARN].to_dict(),
    'artifacts': {
        'backbone_summary_csv': str(summary_path),
        'classification_report_csv': str(report_csv),
        'confusion_matrix_csv': str(cm_csv),
        'confusion_matrix_png': str(cm_png),
        'test_predictions_csv': str(pred_csv),
        'label_map_json': str(label_map_path),
    },
}
summary_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('summary_json:', summary_json)
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# Cell 11 — Optional target ROI smoke inference from existing project crops
SMOKE_CROP_CANDIDATES = [
    PROJECT_ROOT / 'runs' / '_archive' / 'plate_ocr_v1_POCR-EXP-001-target-roi-crops' / 'sample_frames',
    PROJECT_ROOT / 'runs' / 'plate_ocr' / 'POCR-EXP-001-target-roi-crops' / 'sample_frames',
    LOCAL_ROOT / 'runs' / '_archive' / 'plate_ocr_v1_POCR-EXP-001-target-roi-crops' / 'sample_frames',
]

def predict_image(path):
    image = Image.open(path).convert('RGB')
    tensor = eval_tf(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(best_model(tensor), dim=1).cpu().numpy().reshape(-1)
    top = int(np.argmax(probs))
    second = int(np.argsort(probs)[-2]) if len(probs) > 1 else top
    return {
        'crop_uri': str(path),
        'pred_type': id_to_label[top],
        'confidence': float(probs[top]),
        'margin': float(probs[top] - probs[second]),
        'top3': [
            {'label': id_to_label[int(idx)], 'confidence': float(probs[int(idx)])}
            for idx in np.argsort(probs)[-3:][::-1]
        ],
    }

smoke_dir = next((p for p in SMOKE_CROP_CANDIDATES if p.exists() and any(p.glob('*.jpg'))), None)
if smoke_dir is None:
    print('No existing target ROI crop directory found. Smoke inference skipped.')
else:
    print('Running smoke inference on:', smoke_dir)
    smoke_rows = []
    for path in sorted(smoke_dir.glob('*.jpg')):
        row = predict_image(path)
        name = path.name
        row['event_id'] = name.split('_TRK-')[0] if '_TRK-' in name else name
        smoke_rows.append(row)
    smoke_df = pd.DataFrame(smoke_rows)
    smoke_csv = DRIVE_RUN_ROOT / f'{EXP_ID}-local_target_roi_smoke_predictions.csv'
    smoke_df.to_csv(smoke_csv, index=False)
    display(smoke_df.head(20))
    if 'event_id' in smoke_df.columns:
        print('Temporal smoke vote by event:')
        display(smoke_df.groupby(['event_id', 'pred_type']).size().reset_index(name='count').sort_values(['event_id', 'count'], ascending=[True, False]))
    print('smoke_csv:', smoke_csv)


In [ ]:
# Cell 12 — Markdown run report
report_md = DRIVE_RUN_ROOT / f'{EXP_ID}-run_summary.md'
lines = [
    f'# {EXP_ID} Vehicle Type Classifier Summary',
    '',
    f'- Best backbone: `{BEST_BACKBONE}`',
    f'- Best checkpoint: `{BEST_CHECKPOINT}`',
    f'- Validation macro-F1: `{val_eval["macro_f1"]:.4f}`',
    f'- Validation accuracy: `{val_eval["accuracy"]:.4f}`',
    f'- Test macro-F1: `{test_eval["macro_f1"]:.4f}`',
    f'- Test accuracy: `{test_eval["accuracy"]:.4f}`',
    f'- Train/val/test: `{len(train_df)}/{len(val_df)}/{len(test_df)}`',
    '',
    '## FTR Labels',
    '',
    ', '.join(f'`{x}`' for x in FTR_TYPE_LABELS),
    '',
    '## Important Notes',
    '',
    '- This is a dedicated classifier candidate for `arac_bilgisi.tip`.',
    '- Mapping is conservative: ambiguous body styles are skipped instead of forced into wrong labels.',
    '- If any FTR type has low support, add manual/BoxCars/CompCars-derived data before final promotion.',
    '- Runtime promotion still requires target ROI smoke inference and manual review on `Test/video_1-3.mp4`.',
    '',
    '## Class Counts',
    '',
]
for label, count in metadata_df['ftr_type'].value_counts().reindex(FTR_TYPE_LABELS, fill_value=0).items():
    lines.append(f'- `{label}`: {int(count)}')
lines.extend([
    '',
    '## Artifacts',
    '',
    f'- Summary JSON: `{summary_json}`',
    f'- Label map: `{label_map_path}`',
    f'- Classification report CSV: `{report_csv}`',
    f'- Confusion matrix PNG: `{cm_png}`',
])
report_md.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(report_md.read_text())
